# Accessing Spatial Data

<div style="display: flex; justify-content: center;">
    <img src="https://bg.copernicus.org/articles/20/4551/2023/bg-20-4551-2023-avatar-web.png" alt="Credit: Copernicus seagrass mapping from sateillite data example image" style="width: 400px;"/>
    <img src="https://floridakeys.noaa.gov/review/graphics/seagrasswg.jpg" alt="Credit: Seagrass Beds. Florida Keys National Marine Sanctuary." style="width: 400px;"/>
</div>


---

## Overview
This notebook reads jp2 files from satellite imagery in `.SAFE` structure, converting, and saving one scene at a time as netCDF file. 

1. Unzip Downloaded Imagery
2. Sentinel-2 Data Structure
3. Scene by Scene Loop
4. Save Manifest Scene Records

Sentinel-2 satelliete data was collected via [Copernicus Data Space Ecosystem](https://dataspace.copernicus.eu/data-collections/copernicus-sentinel-missions/sentinel-2). Downloaded for the years 2015-2024, around the AOI: Upper Keys, FL (John Pennekamp Coral Reef State Park). Class L1C (post 2016 global product standard, level 2A) was used to download for years 2015-2016, 2017-2024 was downloaded using class L2A (regional product standard).
**Memory Strategy:** jp2 → tif → netCDF → delete, for one scene at a time. No raster data stored as a list by using Pandas, Xarray and Rasterio.


---

In [1]:
# pip install...

### Imports

In [2]:
import os
import zipfile
import re
import gc
import glob
import xarray as xr
import rioxarray as rxr
import rasterio
import numpy as np
import pandas as pd
from rasterio.enums import Resampling

#### Paths and Directories

In [3]:
data_dir = "C:/Users/sulli/DATA_Science/seagrass_project/sentinel2_data"
tif_dir = "band_data_tifs"
processed = "processed"
manifest = os.path.join(processed, "scene_manifest.csv")

band_keys = {"blue": "B02", "green": "B03", "red": "B04", "nir": "B08"}

os.makedirs(tif_dir, exist_ok=True)
os.makedirs(processed, exist_ok=True)

## Unzip Downloaded Files
Scans each year folder for `.zip` files, extracts in place, and then deletes `.zip` to save storage.

In [4]:
for year_folder in sorted(os.listdir(data_dir)):
    year_path = os.path.join(data_dir, year_folder)
    if not os.path.isdir(year_path):
        continue
    
    zips = glob.glob(os.path.join(year_path, "*.zip"))
    if not zips:
        continue

    print(f"{year_folder}: {len(zips)} zip(s) found")

    for zip_path in zips:
        zip_name = os.path.basename(zip_path)
        safe_name = zip_name.replace(".zip", ".SAFE")
        safe_path = os.path.join(year_path, safe_name)

        if os.path.exists(safe_path):
            print(f" already extracted: {safe_name}")
            os.remove(zip_path)
            continue

        print(f" extraction: {zip_name}")
        with zipfile.ZipFile(zip_path, 'r') as z:
            z.extractall(year_path)

        os.remove(zip_path)
        print(f" done, zip deleted")

    print("\nUnzip complete.")
    

---
## Sentinel-2 Data Structure
```
sentinel2_data/
* {year}/
    S2A... .SAFE/
        Granule/
            ...IMG_DATA/
              R10/
                * _B02_10m.jp2
                * _B03_10m.jp2
                * _B04_10m.jp2
                * _B08_10m.jp2
```
| File Name | Spectral Band Color |
|---|---|
| BO4 | Red |
| B02 | Blue |
| B03 | Green |
| B08 | NIR (near infrared) |

## Loop Scene by Scene
Using `for` loops and the function `os.path.join()`to loop scene by scene. 
1. locate 4 bands per `.SAFE` folder
2. Convert each jp2 -> tif file  `.
3. Load 4 tif bands into am `xr.DataSet` with band labelled dimensions and `time`.
4. Save as `processed/{year}/{scene_id}.nc`
5. Delete Dataset from memory and call gc.collect() (to preserve RAM)
6. Record the scene in the manifest list. 

In [ ]:
# run inside a test loop on one SAFE folder
import os
test = glob.glob("C:/Users/sulli/DATA_Science/seagrass_project/sentinel2_data/2016/*.SAFE")[0]
jp2s = glob.glob(os.path.join(test, "**/*.jp2"), recursive=True)
for f in jp2s[:4]:
    print(f"{os.path.basename(f)}: {os.path.getsize(f)/1e6:.1f} MB")

T17RNH_20160302T155402_B01.jp2: 3.0 MB
T17RNH_20160302T155402_B02.jp2: 86.3 MB
T17RNH_20160302T155402_B03.jp2: 83.3 MB
T17RNH_20160302T155402_B04.jp2: 78.8 MB


In [8]:
date_re = re.compile(r"(T\d{2}[A-Z]{3})_(\d{8}T\d{6})")

manifest_records = []

years = sorted(os.listdir(data_dir))
print("Years found:", years)

for year in years:
    year_path = os.path.join(data_dir, year)
    if not os.path.isdir(year_path):
        continue

    safe_folders = sorted(glob.glob(os.path.join(year_path, "*.SAFE")))
    print(f"\n--- {year}: {len(safe_folders)} SAFE folders ---")
    saved_count = 0
    skipped_count = 0

    for safe in safe_folders:

        # find jp2 files inside SAFE
        jp2_files = glob.glob(os.path.join(safe, "**/*.jp2"), recursive=True)

        if not jp2_files:
            print(f"  SKIP (no jp2 found): {os.path.basename(safe)}")
            continue
        
        level = "L1C" if "MSILC" in os.path.basename(safe) else "L2A"
        # L1C: T17RNH_20151113T160052_B02.jp2 (no _10m suffix)
        # L2A: T17RNH_20230515T160511_B02_10m.jp2 (has _10m suffix)

        band_jp2 = {
            "blue":  [f for f in jp2_files if re.search(r"_B02(_10m)?\.jp2$", os.path.basename(f)) and os.path.basename(f).startswith("T")],
            "green": [f for f in jp2_files if re.search(r"_B03(_10m)?\.jp2$", os.path.basename(f)) and os.path.basename(f).startswith("T")],
            "red":   [f for f in jp2_files if re.search(r"_B04(_10m)?\.jp2$", os.path.basename(f)) and os.path.basename(f).startswith("T")],
            "nir":   [f for f in jp2_files if re.search(r"_B08(_10m)?\.jp2$", os.path.basename(f)) and os.path.basename(f).startswith("T")],
        }


        # skip if any band is missing or duplicated
        if not all(len(v) == 1 for v in band_jp2.values()):
            print(f"  SKIP (missing/duplicate bands): {os.path.basename(safe)}")
            print(f"    Found: { {k: len(v) for k, v in band_jp2.items()} }")
            continue

        # parse tile and datetime from B02 filename
        b02_name = os.path.basename(band_jp2["blue"][0])
        m = date_re.search(b02_name)
        if not m:
            print(f"  SKIP (can't parse date): {b02_name}")
            continue

        tile, datetime_str = m.group(1), m.group(2)
        scene_id = f"{tile}_{datetime_str}"
        nc_path  = os.path.join(processed, year, f"{scene_id}.nc")

        # --- step 1: jp2 -> tif ---
        band_tifs = {}
        conversion_ok = True

        for band_name, jp2_list in band_jp2.items():
            jp2_path = jp2_list[0]
            tif_name = os.path.basename(jp2_path).replace(".jp2", ".tif")
            tif_path = os.path.join(tif_dir, tif_name)

            if not os.path.exists(tif_path):
                try:
                    with rasterio.open(jp2_path) as src:
                        profile = src.profile.copy()
                        # preserve CRS, transform, dtype — only change driver
                        profile.update(driver="GTiff", compress="lzw")
                        with rasterio.open(tif_path, "w", **profile) as dst:
                            dst.write(src.read())
                except Exception as e:
                    print(f"  ERROR converting {band_name}: {e}")
                    conversion_ok = False
                    break

            if not os.path.exists(tif_path):
                print(f"  ERROR: tif not created for {band_name}: {tif_path}")
                conversion_ok = False
                break

            band_tifs[band_name] = tif_path

        if not conversion_ok:
            print(f"  SKIP (conversion failed): {scene_id}")
            continue

        # --- step 2: tifs -> xr.Dataset -> netCDF ---
        if not os.path.exists(nc_path):
            os.makedirs(os.path.dirname(nc_path), exist_ok=True)

            arrays = {}
            for band_name, tif_path in band_tifs.items():
                da = rxr.open_rasterio(tif_path, masked=True).squeeze("band", drop=True)
                arrays[band_name] = da.astype(np.float32)

            ds = xr.Dataset(arrays)
            ds = ds.assign_coords(time=pd.to_datetime(datetime_str, format="%Y%m%dT%H%M%S"))
            ds.attrs["scene_id"] = scene_id
            ds.attrs["tile"]     = tile

            encoding = {v: {"zlib": True, "complevel": 4} for v in ds.data_vars}
            ds.to_netcdf(nc_path, encoding=encoding)

            del ds, arrays
            gc.collect()
            status = "saved"
        else:
            status = "skipped"

        manifest_records.append({
            "scene_id": scene_id,
            "tile":     tile,
            "date":     pd.to_datetime(datetime_str, format="%Y%m%dT%H%M%S"),
            "year":     int(year),
            "level":    level,
            "nc_path":  nc_path,
        })

        if status == "saved":
            saved_count += 1
        else:
            skipped_count += 1

    print(f"  saved: {saved_count}  skipped: {skipped_count}")

print("\nDone.")


Years found: ['2015', '2016', '2017', '2018', '2019', '2020', '2021', '2022', '2023', '2024']

--- 2015: 2 SAFE folders ---
  saved: 0  skipped: 2

--- 2016: 9 SAFE folders ---
  saved: 0  skipped: 9

--- 2017: 17 SAFE folders ---
  saved: 5  skipped: 12

--- 2018: 20 SAFE folders ---
  saved: 20  skipped: 0

--- 2019: 18 SAFE folders ---
  saved: 18  skipped: 0

--- 2020: 13 SAFE folders ---
  saved: 13  skipped: 0

--- 2021: 16 SAFE folders ---
  saved: 16  skipped: 0

--- 2022: 17 SAFE folders ---
  saved: 17  skipped: 0

--- 2023: 20 SAFE folders ---
  saved: 20  skipped: 0

--- 2024: 13 SAFE folders ---
  saved: 13  skipped: 0

Done.


## Save Scene Manifest
The scene data manifest catalogs the scene metadata in CSV, from the loop (scene ID, tile, date, year, processing level) and path to each save `.nc` file, for efficient reuse in subsequent notebooks without re-scanning the entire directory.

In [12]:
manifest_df = (
    pd.DataFrame(manifest_records)
    .sort_values(["year", "tile", "date"])
    .reset_index(drop=True)
)

manifest_df.to_csv("processed/scene_manifest.csv", index=False)

print(f"Manifest saved: processed/scene_manifest.csv")
print(f"Total scenes: {len(manifest_df)}")
manifest_df.head(10)

Manifest saved: processed/scene_manifest.csv
Total scenes: 145


,scene_id,tile,date,year,level,nc_path
0,T17RNH_20151113T160052,T17RNH,2015-11-13 16:00:52,2015,L2A,processed\2015\T17RNH_20151113T160052.nc
1,T17RNJ_20151113T160052,T17RNJ,2015-11-13 16:00:52,2015,L2A,processed\2015\T17RNJ_20151113T160052.nc
2,T17RNH_20160302T155402,T17RNH,2016-03-02 15:54:02,2016,L2A,processed\2016\T17RNH_20160302T155402.nc
3,T17RNH_20160630T160512,T17RNH,2016-06-30 16:05:12,2016,L2A,processed\2016\T17RNH_20160630T160512.nc
4,T17RNH_20160720T160512,T17RNH,2016-07-20 16:05:12,2016,L2A,processed\2016\T17RNH_20160720T160512.nc
5,T17RNH_20160918T160512,T17RNH,2016-09-18 16:05:12,2016,L2A,processed\2016\T17RNH_20160918T160512.nc
6,T17RNH_20161127T160512,T17RNH,2016-11-27 16:05:12,2016,L2A,processed\2016\T17RNH_20161127T160512.nc
7,T17RNJ_20160521T160522,T17RNJ,2016-05-21 16:05:22,2016,L2A,processed\2016\T17RNJ_20160521T160522.nc
8,T17RNJ_20160630T160512,T17RNJ,2016-06-30 16:05:12,2016,L2A,processed\2016\T17RNJ_20160630T160512.nc
9,T17RNJ_20160720T160512,T17RNJ,2016-07-20 16:05:12,2016,L2A,processed\2016\T17RNJ_20160720T160512.nc


### Surface Level Test
Confirmation that one scene opens without errors and the structure of the raster data (varaibles, dimensions, coordinates) are stored in the DataSet.

In [13]:
sample_path = manifest_df.iloc[0]["nc_path"]
ds_check = xr.open_dataset(sample_path)
print(ds_check)
ds_check.close()

<xarray.Dataset> Size: 2GB
Dimensions:      (y: 10980, x: 10980)
Coordinates:
  * y            (y) float64 88kB 2.8e+06 2.8e+06 2.8e+06 ... 2.69e+06 2.69e+06
  * x            (x) float64 88kB 5e+05 5e+05 5e+05 ... 6.098e+05 6.098e+05
    spatial_ref  int64 8B ...
    time         datetime64[ns] 8B ...
Data variables:
    blue         (y, x) float32 482MB ...
    green        (y, x) float32 482MB ...
    red          (y, x) float32 482MB ...
    nir          (y, x) float32 482MB ...
Attributes:
    scene_id:  T17RNH_20151113T160052
    tile:      T17RNH


---

## Summary
Outputs of this notebook include:
- `band_data_tifs/` converted from jp2 for reuse.
- `processed/{year}/{scene_id}.nc` containing one DataSet per scene with bands `blue`, `green`, `red`, and `nir`, and `time` coordinate dimensions.
- `processed/scene_manifest.csv` which catalogs the scenes for storage efficiency.

**Next:** `02_preprocess_imagery` to load manifest, apply cloud mask, clip to AOIm and save clipped bands.

## Resources and references

- GeeksforGeeks. (2026, January 13). Understanding python pickling with example. https://www.geeksforgeeks.org/python/understanding-python-pickling-example/ 
- The Python Software Foundation. (2026, March 20). Glob - unix style pathname pattern expansion. Python documentation. https://docs.python.org/3/library/glob.html
- Project Pythia Notebook template